# FIT5202 Assignment 2A : Building ML Models for Smart Transport & Predictive Parking

## Table of Contents
*  
    * [Part 1 : Data Loading, Transformation and Exploration](#part-1)
    * [Part 2 : Feature extraction and ML training](#part-2)
    * [Part 3 : Hyperparameter Tuning and Model Optimisation](#part-3)  
Please add code/markdown cells as needed.

In [1]:
import sys
import pyspark
import pandas
import pyarrow

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)

Python: 3.10.12 (main, Nov  4 2025, 08:48:33) [GCC 11.4.0]
PySpark: 4.1.0
Pandas: 2.3.3
PyArrow: 22.0.0


In [2]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

conf = (
    SparkConf()
    .setAppName("FIT5202-A2A-Environment-Test")
    .setMaster("local[*]")
    .set("spark.sql.files.maxPartitionBytes", str(32 * 1024 * 1024))
)

spark = (
    SparkSession.builder
    .config(conf=conf)
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print(
    "Maximum partition bytes:",
    spark.conf.get("spark.sql.files.maxPartitionBytes")
)

spark.range(10).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/12 17:15:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.0
Spark master: local[*]
Maximum partition bytes: 33554432
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



Part 1: Data Loading, Transformation, and Exploration (30%)
## 1.1 Data Loading (2%)
In this section, you must load the given datasets into PySpark DataFrames and use DataFrame functions to process the data. For plotting, various visualisation packages can be used, but please ensure that you have included instructions to install the additional packages and that the installation will be successful in the provided Docker container (in case your marker needs to clear the notebook and rerun it).

### 1.1.1 Data Loading <a class="anchor" name="1.1"></a>
SparkSession Configuration: Write the code to initialise a SparkSession. You must use a SparkConf object to set a descriptive application name, ensure the maximum partition byte size does not exceed 32MB, and configure Spark to execute locally using all available CPU cores.

1.1.2 Schema Definition: Explicitly define StructType schemas for a2_traffic_data.csv, a2_parking_sensor.csv, adhering to the data types specified in metadata.xlsx. Do not rely on schema inference such as inferSchema.

1.1.3 DataFrame Ingestion: Load the CSV files into distinct PySpark DataFrames using your defined schemas and print their schemas to confirm correct types.

### 1.2 Data Transformation and Feature Creation (14%) <a class="anchor" name="1.2"></a>
Please refer to the specification for backgroud.  
1.2.1 Spatial-Temporal Join & Aggregation: Based on the domain knowledge discussed above, develop a join strategy to join the traffic counts and parking availability data frames, then write PySpark code to join the traffic and parking datasets. Your resulting consolidated DataFrame must isolate available_spots as the continuous numerical value you will use as the target label.

1.2.2 Perform imputation

1.2.3 Feature Engineering

In [ ]:
# create feature 1


In [ ]:
# create another feature

### 1.3 Exploratory Data Analysis (14%) <a class="anchor" name="1.3"></a>

1.	Statistical Summary: Using your prepared DataFrame, generate a structured statistical overview:
For all numerical features, output count, mean, standard deviation, min, max, 25th percentile, 50th percentile (median), and 75th percentile. (Note: numerical features do not include IDs, just those columns related to the analysis.)
For categorical/string features, display the top 5 most frequent categories along with their counts.

2.	Bivariate and Multivariate Visualisations: Write code to produce two distinct bivariate or multivariate plots analysing interactions within the data (e.g., Traffic volume vs. Available Parking Spots, or Average Speed vs. Occupancy Rates during Peak vs. Off-Peak Hours).
Provide an analytical discussion beneath each figure (maximum 150 words per plot) detailing the observed trend and its practical implications for model training.  
You can refer to https://samplecode.link  


## Part 2. Feature extraction and ML training <a class="anchor" name="part-2"></a>
In this section, you must use PySpark DataFrame functions and ML packages for data preparation, model building, and evaluation. Other ML packages, such as scikit-learn, should not be used to process the data; however, it’s fine to use them to display the result or evaluate your model.  
### 2.1 Feature Selection & Justification (10%)
Review the columns across the joined datasets. In a markdown cell, provide a structured discussion (maximum 300 words):  
- Identify which specific columns represent noise or distractors for the target label and explain why they should be excluded.  
- Justify the predictive features you have selected for your feature vector.  
- Detail the transformations required (e.g., index encoding, one-hot encoding, vector scaling) before vector assembly.  
Write the PySpark DataFrame code to assemble your filtered feature set.  


### 2.2 Preparing Spark ML Pipelines (10%)  <a class="anchor" name="2.2"></a>

Construct the required data transformation stages using PySpark MLlib:  
- StringIndexer and OneHotEncoder for non-ordinal categorical features.  
- VectorAssembler to combine all numerical and encoded categorical columns into a single features vector.  
- Perform scaling and normalisation as appropriate.  
(Note: If you have already created those transformations, you can use them directly in the pipelines.)  
Define two ML Estimators configured for continuous label regression:  
- Pipeline 1: RandomForestRegressor  
- Pipeline 2: GBTRegressor (Gradient-Boosted Tree Regressor)  
Assemble each estimator with your feature transformation stages into two clean, end-to-end pipeline objects. Do not call .fit() or .transform() at this stage.  


### 2.3 2.3 Model Training and Evaluation (15%)

2.3.1 Data Splitting: Split your consolidated dataset into training (70%) and testing (30%) subsets using seed=2026.

2.3.2 Metric Discussion: Discuss the evaluation metrics you will use to evaluate your regression models (e.g., Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), R^2, etc.). Explain why you chose those metrics over others.

2.3.3 Pipeline Fitting & Comparison: Train both pipelines on the training split. Evaluate their performance on the unseen test split using RegressionEvaluator.  
Present a side-by-side comparison table of the evaluation metrics, analyse why one algorithm outperformed the other, and save the superior pipeline model to disk. (You will load this model in Assignment 2B).  
(Note: You may need to go through a few training loops or use more data to create a better-performing model.)

### Part 3: Performance Tuning (35%) <a class="anchor" name="part-3"></a>  
Apply the techniques you have learnt from the labs, for example, CrossValidator, TrainValidationSplit, ParamGridBuilder, etc., to perform further hyperparameter tuning and model optimisation.  
The assessment is based on the quality of your work/process, not the quality of your model. Please include your thoughts/ideas/discussions.

Using the superior model pipeline identified in Part 2, perform systematic hyperparameter tuning to optimise predictive accuracy.  
1. Search Grid & Cross-Validation: Construct a parameter grid, and select at least two hyperparameters to tune (e.g., maxDepth, numTrees/maxIter, or stepSize/minInstancesPerNode).


2. Configure a CrossValidator with either k-fold cross-validation or TrainValidationSplit, using the metrics you suggested as the primary optimisation metric.


3. Process Documentation & Analysis:
In a markdown cell, document:  
- The parameters and value ranges tested in your grid.  
- The computational cost and observed convergence behaviour across folds.  
- The quantitative performance gain achieved relative to your baseline model from Part 2.  

In [ ]:
4. Model Export: Save your finalised, tuned pipeline model to an export directory (e.g., models/best_tuned_parking_model).  
This persisted model will be directly deployed into your real-time Kafka streaming consumer in Assignment 2B.  
(note: If you don't complete the optimisation part, you can also use the model from part 2 in A2B.)

## References:
Please add your references below: